In [8]:
%pip install Pillow

   ━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.5/7.1 MB 44.0 kB/s eta 0:02:31
Resuming download pillow-13.0.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (524 kB/7.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 1.8 MB/s  0:00:03 eta 0:00:010m
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import base64
import os
import requests
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
from PIL import Image as PILImage
from IPython.display import display, HTML

In [10]:
load_dotenv("../.env") # load our env vars
client = OpenAI(
    base_url="https://api.meta.ai/v1",
    api_key=os.environ["MODEL_API_KEY"],
)

In [11]:
def generate(prompt, filename, previous=None, size="1536x1024"):
    output = Path("ps1_scenes")
    output.mkdir(exist_ok=True)

    kwargs = {
        "model": "muse-image-1.0",
        "input": prompt,
        "tools": [{
            "type": "image_generation",
            "size": size,
            "output_format": "webp",
        }],
    }

    if previous is not None:
        kwargs["previous_response_id"] = previous.id

    response = client.responses.create(**kwargs)

    image_item = next(
        (item for item in response.output
         if item.type == "image_generation_call" and item.result),
        None,
    )

    if image_item is None:
        raise RuntimeError(f"No image generated: {response.status}")

    path = output / filename
    path.write_bytes(base64.b64decode(image_item.result))

    print(f"Saved: {path}")
    display(PILImage.open(path))

    return response

In [12]:
style_prompt = """
Create a reference image establishing the visual style
of a late-1990s PlayStation 1 survival horror video game.

Scene:
An abandoned railway platform in a small Northern Ontario town.
An old brick railway building, metal railings,
weathered platform, railway tracks disappearing into fog.

Visual style:
- Authentic late-1990s PS1 graphics
- Low polygon counts and angular geometry
- Low-resolution pixelated textures
- Affine texture warping
- Vertex lighting
- Visible dithering and limited color gradients
- Muted blue-gray and brown color palette
- Thick atmospheric distance fog
- Fixed-camera survival horror aesthetic

Composition:
A cinematic wide shot with convincing real-world proportions.

Avoid modern photorealism, smooth geometry, modern game graphics,
illustration styling, excessive bloom, or modern ray tracing.

No characters, no HUD, no lettering.

Treat this image as the MASTER STYLE REFERENCE
for all later generations.
"""

style = generate(
    style_prompt,
    "01_style.webp",
)

Saved: ps1_scenes/01_style.webp


AttributeError: type object 'Image' has no attribute 'open'

In [ ]:
character_prompt = """
Create a NEW CHARACTER REFERENCE SHEET.

This is a separate asset, not another view of the railway platform.

Character:
A man in his late 30s with medium-length dark brown hair,
average build, wearing a red-and-navy plaid flannel jacket,
blue jeans and worn brown work boots.

Produce a clean character turnaround showing:
1. Full-body front view
2. Full-body side view
3. Full-body back view

All three views must show the SAME character with:
- Identical facial features and hairstyle
- Identical clothing and plaid pattern
- Identical height and body proportions
- Consistent shoes, sleeves and jacket details

Neutral standing pose with arms slightly away from body.
Plain neutral gray background, even studio lighting.

Use clean realistic character concept-art rendering for now.
Do not apply the PS1 graphics treatment yet.

No environment, props or other characters.

This is the MASTER CHARACTER REFERENCE.
"""

character = generate(
    character_prompt,
    "02_character.webp",
    previous=style,
)

In [ ]:
interior_prompt = """
Create a NEW INTERIOR ENVIRONMENT REFERENCE.

This is a separate image of an abandoned railway waiting room.

Room geometry:
- Rectangular room, approximately 8 meters by 6 meters
- Entrance on the south wall
- Green exit door centered on the north wall
- Old wooden ticket counter against the west wall
- Two tall windows along the east wall
- One wooden bench below the east windows
- Worn checkerboard floor tiles
- Fluorescent ceiling lights
- Peeling cream-colored walls

Camera:
Wide architectural view from the southwest corner,
looking northeast across the room.

Show enough of the room to establish its geometry,
door positions, furniture and wall relationships.

Use a clean architectural visualization style for now.
Do not apply PS1 graphics yet.

No characters.

IMPORTANT:
This exact room layout must be maintained
in all subsequent images and camera angles.

This is the MASTER ENVIRONMENT REFERENCE.
"""

interior = generate(
    interior_prompt,
    "03_interior.webp",
    previous=character,
)

In [ ]:
stylize_character_prompt = """
Return to the MASTER CHARACTER REFERENCE created earlier.

Restyle that exact character using the MASTER PS1 STYLE REFERENCE.

Keep:
- Same person and facial appearance
- Same red-and-navy plaid flannel jacket
- Same jeans and brown boots
- Same body proportions
- Same front, side and back turnaround views

Apply:
- Late-1990s PS1 low-polygon character geometry
- Angular face, hands, arms and clothing
- Low-resolution pixelated clothing textures
- Simple vertex shading
- Limited texture detail
- Authentic survival horror character-model aesthetic

This should look like an actual in-game PS1 character model,
not a modern illustration imitating retro pixel art.

Use a plain gray background.
No room or environment.

Output a clean three-view character reference sheet.
"""

ps1_character = generate(
    stylize_character_prompt,
    "04_ps1_character.webp",
    previous=interior,
)

In [ ]:
stylize_interior_prompt = """
Return to the MASTER INTERIOR ENVIRONMENT REFERENCE.

Recreate that exact room using the PS1 STYLE REFERENCE.

Preserve the original room's geometry:
- Entrance on south wall
- Green exit door on north wall
- Ticket counter on west wall
- Two windows and wooden bench on east wall
- Checkerboard floor
- Fluorescent ceiling lights

Apply:
- Authentic late-1990s PS1 graphics
- Low-polygon furniture and architecture
- Low-resolution textures
- Texture warping and dithering
- Vertex lighting
- Muted survival-horror color grading
- Dark corners and subtle atmospheric fog

Maintain the room's real-world proportions.
Use the same camera angle as the original interior reference.

No characters.
No HUD.
No text.

This is the FINAL STYLIZED ENVIRONMENT REFERENCE.
"""

ps1_interior = generate(
    stylize_interior_prompt,
    "05_ps1_interior.webp",
    previous=ps1_character,
)

In [ ]:
camera_sheet_prompt = """
Create a six-panel cinematography reference sheet.

Use the established PS1 CHARACTER and the established
PS1 INTERIOR ENVIRONMENT.

The SAME character must appear in the SAME room
in all six panels.

Camera variations:

A. ESTABLISHING SHOT
Wide view from the southwest corner.
Character standing near the center of the room.

B. THIRD-PERSON SHOT
Camera behind the character at shoulder height.
Character facing the green north exit door.

C. SIDE PROFILE
Camera near the east windows.
Character walking toward the ticket counter.

D. OVER-THE-SHOULDER
Camera close behind the character's right shoulder.
Green exit door is the focal point.

E. LOW-ANGLE SHOT
Camera close to the checkerboard floor.
Character standing underneath fluorescent lighting.

F. CLOSE-UP
Close cinematic view of the character's hand
reaching for the green exit door handle.

REQUIREMENTS:
- All panels use identical PS1 graphics.
- Same character appearance and outfit.
- Same room architecture and furniture arrangement.
- Plausible character scale relative to the room.
- Different actual camera positions, not cropped duplicates.
- Maintain believable spatial relationships.

Arrange six equally sized panels in a 3-column,
2-row storyboard grid.

Label each panel A through F.
No HUD, dialogue or additional text.
"""

camera_sheet = generate(
    camera_sheet_prompt,
    "06_camera_sheet.webp",
    previous=ps1_interior,
)

In [ ]:
shots = {
    "A_establishing": """
        Render camera shot A from the storyboard.
        Wide establishing shot from southwest corner.
        Character standing near the center of the waiting room.
        Show the room's overall layout and atmosphere.
    """,

    "B_third_person": """
        Render camera shot B from the storyboard.
        Third-person camera directly behind the character.
        Camera at shoulder height.
        Character facing the north exit door.
        Use believable player-to-environment scale.
    """,

    "C_side_profile": """
        Render camera shot C from the storyboard.
        Side-profile shot from near the east windows.
        Character walking toward the western ticket counter.
        Keep correct room geometry and proportions.
    """,

    "D_over_shoulder": """
        Render camera shot D from the storyboard.
        Tight over-the-right-shoulder composition.
        The green exit door is the focal point.
        Keep the shoulder and door perspective natural.
    """,

    "E_low_angle": """
        Render camera shot E from the storyboard.
        Camera near the checkerboard floor.
        Low-angle view looking upward at the character.
        Fluorescent lighting visible overhead.
    """,

    "F_closeup": """
        Render camera shot F from the storyboard.
        Cinematic close-up of the character's hand
        reaching toward the green exit door handle.
        Maintain consistent clothing and hand proportions.
    """,
}

In [ ]:
renders = {}

for name, shot_prompt in shots.items():

    prompt = f"""
    Generate ONE standalone cinematic game screenshot.

    {shot_prompt}

    Use the established PS1 character, interior,
    art style and six-panel camera reference sheet.

    Reproduce the requested camera composition
    as a full-frame standalone render.

    Maintain:
    - Identical character design
    - Identical room geometry
    - Accurate character scale
    - Consistent textures and lighting
    - Authentic low-poly PS1 graphics

    Do not generate a contact sheet.
    Do not include panel labels.
    No HUD, borders or text.
    """

    renders[name] = generate(
        prompt,
        f"07_{name}.webp",
        previous=camera_sheet,
    )

In [ ]:
refined_B = generate(
    """
    Refine this third-person gameplay shot.

    The character is currently too large relative
    to the surrounding architecture.

    Correct the character's scale so an average-height
    adult appears proportionate to the doors,
    windows, ceiling and furniture.

    Keep the existing camera angle, outfit,
    room layout, textures and PS1 graphics.

    Do not redesign the room or character.
    """,
    "08_refined_B.webp",
    previous=renders["B_third_person"],
)